# Track 3 · 02 EEGNet — the organisers' sleep-onset regressor, on the harness's own windows

**One job.** Train the organisers' EEGNet baseline on the warm-up data, check it learns,
score it the way the leaderboard does, and package it for submission.

Self-contained: upload this one file. The windows come from `track3_01_prepare` on Drive.

---

## The recipe, from source

Model and loss are `tracks/sleep_onset/solvers/eegnet_reg.py`: `braindecode.models.EEGNet`
with one output, MSE on the latency divided by 600, Adam at 1e-3, batch 64; predictions
are multiplied back by 600 and clamped to [0, 600] s.

Two things come from the neuralbench task config instead, because the baseline file
trains for a fixed 20 epochs with plain shuffling:

- **Bin-balanced sampling** (`RegressionBinSampler`). About three quarters of the windows
  sit in the 300–600 s bin, most of them at the 600 s cap, so plain shuffling teaches the
  model to say "600". Each latency bin is drawn equally often instead.
- **Up to 40 epochs, keep the best on validation bMAE, stop after 7 without improvement.**

## Scoring

- **bMAE**: the warm-up leaderboard metric, copied from `benchmark_utils/metrics.py`. MAE
  inside each bin (0–40, 40–90, 90–300, 300–600 s), then the plain mean of the four.
- **W-bMAE**: the sealed-phase metric, weighted 10, 5, 3, 1 so being close to onset counts
  most. Its scorer is not released yet; the formula here (weighted mean of the per-bin
  MAEs) is our reading of the announcement.

Lower is better for both. Reference points on the warm-up:

| | bMAE |
|---|---|
| predict 0 s everywhere (the leaderboard floor) | 205.42 |
| EEGNet, published NeuralBench reference | 143.30 |
| REVE, published NeuralBench reference | 134.89 |
| warm-up leader when last checked | 83.44 |

The test set here is the harness's: the same 16 people the warm-up scores. Predicting 0
everywhere on it should give 205.42, printed below as a check on the window and target
rule. It cannot check the split: every night onsets more than 20 minutes in, so every
night has the same 240 targets and any 16 people give the same floor.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

# eegnet_reg.py's values, plus neuralbench's sampler and early stopping.
EPOCHS = 40
PATIENCE = 7          # epochs without a better validation score before stopping
MONITOR = 'bmae'      # 'bmae' (warm-up metric, neuralbench's monitor) or 'wbmae'
BATCH = 64
LR = 1e-3
DROPOUT = 0.25        # EEGNet's default, which eegnet_reg.py uses
WEIGHT_DECAY = 0.0
SAMPLER = 'bins'      # 'bins': each latency bin equally often; 'shuffle': plain shuffling

CAP_S = 600.0
BIN_EDGES = (0.0, 40.0, 90.0, 300.0, 600.0)
BIN_WEIGHTS = (10.0, 5.0, 3.0, 1.0)   # sealed-phase W-bMAE

RUN_CONTROL = True    # positive control first, so a flat result is readable
MAKE_ZIP = True
SEED = 33

RUN_TAG = f'd{DROPOUT}_b{BATCH}_{SAMPLER}_{MONITOR}'
print(f'NOTEBOOK VERSION: track3_02 v1   settings {RUN_TAG}   epochs {EPOCHS}')

In [ ]:
%pip install -q 'braindecode>=1.8'

import importlib
for mod in ('braindecode', 'torch', 'numpy'):
    print(f'{mod:12} {importlib.import_module(mod).__version__}')

In [ ]:
import json, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE = Path(ROOT) / 'Track3'
DATA = BASE / 'sleep_edf'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

man = json.loads((DATA / 'manifest.json').read_text())
parts = {s: ([], [], []) for s in ('train', 'val', 'test')}   # X, y, person
CH = None
for rec, r in sorted(man['recordings'].items()):
    if not r.get('n_windows'):
        continue
    z = np.load(DATA / 'windows' / r['file'])
    p = parts[r['split']]
    p[0].append(z['X'].astype('float32')); p[1].append(z['y'].astype('float32'))
    p[2].append(np.full(len(z['y']), r['subject']))
    CH = CH or r['channels']
X, Y, P = {}, {}, {}
for s, (xs, ys, ps) in parts.items():
    X[s], Y[s], P[s] = np.concatenate(xs), np.concatenate(ys), np.concatenate(ps)
N_CHANS, N_TIMES = X['train'].shape[1:]
SFREQ = man['preprocessing']['sfreq']
assert not set(P['test']) & set(P['train']) and not set(P['val']) & set(P['train'])


def bin_of(y):
    return np.digitize(y, BIN_EDGES[1:-1])


for s in ('train', 'val', 'test'):
    print(f'{s:5}  {len(set(P[s])):2d} people  {len(Y[s]):6,d} windows   '
          f'per bin {np.bincount(bin_of(Y[s]), minlength=4).tolist()}')
print(f'channels {CH}, {N_TIMES} samples at {SFREQ:g} Hz, device {DEVICE}')


def binned_mae(y_true, y_pred, bin_edges=BIN_EDGES):
    """benchmark_utils/metrics.py, verbatim: the warm-up leaderboard metric."""
    y_true = np.asarray(y_true, dtype=np.float64).ravel()
    y_pred = np.asarray(y_pred, dtype=np.float64).ravel()
    edges = np.asarray(bin_edges, dtype=np.float64)
    err = np.abs(y_pred - y_true)
    idx = np.digitize(y_true, edges[1:-1], right=False)
    in_range = (y_true >= edges[0]) & (y_true <= edges[-1])
    maes = [err[in_range & (idx == b)].mean()
            for b in range(len(edges) - 1)
            if np.any(in_range & (idx == b))]
    return float(np.mean(maes)) if maes else float('nan')


def per_bin(y_true, y_pred):
    b = bin_of(y_true)
    return [float(np.abs(y_pred - y_true)[b == k].mean()) if (b == k).any() else np.nan
            for k in range(len(BIN_EDGES) - 1)]


def w_bmae(y_true, y_pred):
    """Our reading of the sealed-phase W-bMAE: per-bin MAE, weighted 10/5/3/1."""
    m = np.array(per_bin(y_true, y_pred)); w = np.array(BIN_WEIGHTS)
    ok = ~np.isnan(m)
    return float((m[ok] * w[ok]).sum() / w[ok].sum())


def scores(y_true, y_pred):
    return dict(bmae=binned_mae(y_true, y_pred), wbmae=w_bmae(y_true, y_pred),
                mae=float(np.abs(y_pred - y_true).mean()))


def null(y_true, y_pred, n=200):
    """bMAE with the predictions shuffled across windows: same predictions, no link
    to the targets. A model has learned something only if it beats this."""
    return np.array([binned_mae(y_true, rng.permutation(y_pred)) for _ in range(n)])


floor = binned_mae(Y['test'], np.zeros_like(Y['test']))
print(f'\npredict 0 s on our test windows: bMAE {floor:.2f}   (leaderboard floor 205.42; '
      f'{"match: windows and targets follow the harness rule" if abs(floor - 205.42) < 0.01 else "NO MATCH: check the windows and targets"}'
      f')')

In [ ]:
from braindecode.models import EEGNet


def bin_balanced_weights(y):
    """RegressionBinSampler: inverse-frequency weights per bin, so each populated bin
    gets the same expected share of every epoch."""
    b = bin_of(y)
    counts = np.bincount(b, minlength=len(BIN_EDGES) - 1).astype('float64')
    return torch.tensor(1.0 / counts[b], dtype=torch.float64)


def train(Xtr, ytr, Xv, yv, epochs, label, quiet=False):
    net = EEGNet(n_chans=N_CHANS, n_outputs=1, n_times=N_TIMES,
                 drop_prob=DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    Xtr_t, ytr_t = torch.tensor(Xtr), torch.tensor(ytr) / CAP_S
    Xv_t = torch.tensor(Xv)
    w = bin_balanced_weights(ytr)
    gen = torch.Generator().manual_seed(SEED)

    @torch.no_grad()
    def infer(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        out = torch.cat([net(Xa[i:i + 1024].to(DEVICE)).squeeze(-1)
                         for i in range(0, len(Xa), 1024)])
        return (out * CAP_S).clamp(0, CAP_S).cpu().numpy()

    if not quiet:
        print(f'{label}: EEGNet {sum(p.numel() for p in net.parameters()):,} parameters, '
              f'{len(ytr):,} training windows, sampler {SAMPLER}')
    best, best_state, hist, since = np.inf, None, [], 0
    for ep in range(1, epochs + 1):
        net.train()
        order = (torch.multinomial(w, len(w), replacement=True, generator=gen)
                 if SAMPLER == 'bins' else torch.randperm(len(ytr), generator=gen))
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            pred = net(Xtr_t[b].to(DEVICE)).squeeze(-1)
            loss = torch.nn.functional.mse_loss(pred, ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        pv = infer(Xv_t)
        sc = scores(yv, pv)
        vl = float(np.mean(((pv - yv) / CAP_S) ** 2))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc[MONITOR] < best
        if better:
            best, since = sc[MONITOR], 0
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            since += 1
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bMAE {sc["bmae"]:6.2f}  W-bMAE {sc["wbmae"]:6.2f}  '
                  f'MAE {sc["mae"]:6.2f}{"  *" if better else ""}  {time.time() - t0:.0f}s')
        if since >= PATIENCE:
            if not quiet:
                print(f'  stopped: no better val {MONITOR} for {PATIENCE} epochs')
            break
    net.load_state_dict(best_state)
    return net, hist, infer


print('ready')

## Step 1 — positive control

A pattern whose size grows with the target is planted into a copy of the windows, and the
same training loop runs on it. If this does not learn, the loop is broken and Step 2 means
nothing. If it does, a flat Step 2 is a statement about the data.

In [ ]:
if RUN_CONTROL:
    t_ax = np.arange(N_TIMES) / SFREQ
    template = np.sin(2 * np.pi * 2.0 * t_ax).astype('float32')     # 2 Hz, fixed phase

    def plant(Xa, ya):
        return (Xa + 2.0 * (ya / CAP_S)[:, None, None] * template[None, None, :]).astype('float32')

    sub = rng.choice(len(Y['train']), min(8000, len(Y['train'])), replace=False)
    _, hist_c, infer_c = train(plant(X['train'][sub], Y['train'][sub]), Y['train'][sub],
                               plant(X['val'], Y['val']), Y['val'], 15, 'control', quiet=True)
    best_c = min(h['bmae'] for h in hist_c)
    null_c = null(Y['val'], infer_c(plant(X['val'], Y['val'])), n=50)
    print(f'control val bMAE: epoch 1 {hist_c[0]["bmae"]:.2f} -> best {best_c:.2f}   '
          f'(shuffled-prediction null {null_c.mean():.2f})')
    print('CONTROL PASSES: the loop learns a planted signal' if best_c < 0.5 * null_c.mean() else
          'CONTROL FAILS: the loop cannot learn a planted signal; fix before Step 2')
else:
    print('control skipped')

## Step 2 — the real run

What to look for is validation bMAE falling below its own null (the same predictions
shuffled across windows) and the best epoch coming after the first.

In [ ]:
net, hist, infer = train(X['train'], Y['train'], X['val'], Y['val'], EPOCHS, 'real')
best = min(hist, key=lambda h: h[MONITOR])
print(f'\nbest val {MONITOR} {best[MONITOR]:.2f} at epoch {best["epoch"]} of {len(hist)}')

pv = infer(X['val'])
nullv = null(Y['val'], pv)
thr = float(np.percentile(nullv, 5))
drop = hist[0]['bmae'] - best['bmae']
print(f'val bMAE {best["bmae"]:.2f} against shuffled predictions: mean {nullv.mean():.2f}  '
      f'sd {nullv.std():.2f}  5th pct {thr:.2f}')
if nullv.std() < 1e-6:
    print('The null has no spread: the model predicts (nearly) one value for every window.')
TRENDING = bool(best['bmae'] < thr and best['epoch'] > 1 and drop > 2 * nullv.std())
print('Trending: validation improved past epoch 1 and beats its null.' if TRENDING else
      'Above the null but no clear improvement after epoch 1.' if best['bmae'] < thr else
      'Not trending: validation never beats its own null.')

pt = infer(X['test'])
st = scores(Y['test'], pt)
med = float(np.median(Y['train']))
rows = [('EEGNet (this run)', st, per_bin(Y['test'], pt)),
        ('predict 0 s', scores(Y['test'], np.zeros_like(pt)), per_bin(Y['test'], np.zeros_like(pt))),
        (f'predict {med:.0f} s (train median)', scores(Y['test'], np.full_like(pt, med)),
         per_bin(Y['test'], np.full_like(pt, med)))]
print(f'\ntest set (the warm-up people)          bMAE   W-bMAE    MAE    per-bin MAE '
      f'0-40 / 40-90 / 90-300 / 300-600')
for name, s_, pb in rows:
    print(f'  {name:<34} {s_["bmae"]:6.2f}  {s_["wbmae"]:6.2f}  {s_["mae"]:6.2f}    '
          + ' / '.join(f'{v:5.1f}' for v in pb))
print('  published: floor 205.42, EEGNet 143.30, REVE 134.89, leader 83.44 (bMAE)')

torch.save(net.state_dict(), WEIGHTS / f'sleep_eegnet_{RUN_TAG}.pt')
(WEIGHTS / f'sleep_eegnet_{RUN_TAG}.json').write_text(json.dumps(dict(
    run_tag=RUN_TAG, dropout=DROPOUT, batch=BATCH, sampler=SAMPLER, monitor=MONITOR,
    best_epoch=best['epoch'], test=st, history=hist), indent=1))
print(f'\nsaved -> {WEIGHTS / f"sleep_eegnet_{RUN_TAG}.pt"}')

## Step 3 — package for submission

The zip is `weights.pt`, the solver and a small config. The solver matches incoming
electrodes to the training ones **by name** and prints how many matched into the
Codabench log. Before zipping, it is loaded back the way the harness loads it and checked:
its predictions must equal this notebook's, and reordering or adding electrodes must not
change them.

In [ ]:
# Source of truth is solvers/track3_sleep_eegnet.py in the repo.
SUBMISSION_PY = "\"\"\"Track 03 submission: EEGNet regressing seconds to sleep onset.\n\nThe model and loss follow the organisers' `eegnet_reg.py` baseline: one output,\nMSE on the latency divided by 600, predictions multiplied back and clamped to\n[0, 600] s. The weights come from `track3_02_eegnet.ipynb`, trained on\nSleep-EDF windows made with the harness's own preprocessing.\n\nInputs are matched to the training electrodes by name (Fpz-Cz, Pz-Oz on the\nwarm-up data): extra electrodes are ignored, missing ones are filled with zeros,\nand it raises if none match. The sealed phase uses a Muse headband with\ndifferent electrodes, so this file serves the warm-up only.\n\"\"\"\n\nimport json\n\nimport torch\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import EEGNet\n\n\ndef resample_fft(x, n_out):\n    \"\"\"Resample along time via the rfft spectrum. Pure torch, since nothing is\n    installed at evaluation.\"\"\"\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass ChannelMatchedRegressor:\n\n    def __init__(self, cfg, in_names, in_sfreq, device):\n        self.device = device\n        self.cap = float(cfg[\"cap_s\"])\n        self.n_times = int(cfg[\"n_times\"])\n        self.sfreq = float(cfg[\"sfreq\"])\n        self.in_sfreq = float(in_sfreq)\n        self.net = EEGNet(\n            n_chans=len(cfg[\"channels\"]), n_outputs=1, n_times=self.n_times,\n            drop_prob=float(cfg[\"dropout\"]),\n        ).to(device)\n\n        lookup = {name.casefold(): i for i, name in enumerate(in_names)}\n        self.src = [lookup.get(name.casefold(), -1) for name in cfg[\"channels\"]]\n        missing = [n for n, i in zip(cfg[\"channels\"], self.src) if i < 0]\n        print(\n            f\"[EEGNet-sleep] matched {len(self.src) - len(missing)}/{len(self.src)} \"\n            f\"training channels by name among {len(in_names)} inputs; \"\n            f\"zero-filled: {missing if missing else 'none'}\"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(\n                f\"no training channel found in the input: {list(in_names)[:8]}\"\n            )\n\n    @torch.no_grad()\n    def predict(self, X):\n        self.net.eval()\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        # Match the training rate and window length. At the harness's own\n        # settings both are no-ops.\n        if self.in_sfreq != self.sfreq:\n            out = resample_fft(\n                out, int(round(out.shape[-1] * self.sfreq / self.in_sfreq))\n            )\n        if out.shape[-1] > self.n_times:\n            out = out[..., : self.n_times]\n        elif out.shape[-1] < self.n_times:\n            out = F.pad(out, (0, self.n_times - out.shape[-1]))\n        return (self.net(out).squeeze(-1) * self.cap).clamp(0.0, self.cap)\n\n\nclass Solver(CompetSolver):\n\n    name = \"EEGNet-sleep-onset\"\n\n    requirements = [\"pip::braindecode\"]\n\n    def load_model(self, meta):\n        sub = meta[\"submission_dir\"]\n        cfg = json.loads((sub / \"sleep_eegnet_config.json\").read_text())\n        in_names = meta.get(\"ch_names\") or [c[\"ch_name\"] for c in meta[\"chs_info\"]]\n        model = ChannelMatchedRegressor(cfg, in_names, meta[\"sfreq\"], meta[\"device\"])\n        state = torch.load(\n            sub / \"weights.pt\", map_location=meta[\"device\"], weights_only=True\n        )\n        model.net.load_state_dict(state)\n        return model\n"

In [ ]:
if MAKE_ZIP and not (TRENDING or globals().get('FORCE_ZIP')):
    print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')
elif MAKE_ZIP:
    import importlib.util, shutil, sys, types
    out = BASE / 'submissions' / f'track3_eegnet_{RUN_TAG}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    torch.save({k: v.cpu() for k, v in net.state_dict().items()}, out / 'weights.pt')
    (out / 'sleep_eegnet_config.json').write_text(json.dumps(dict(
        channels=list(CH), n_times=int(N_TIMES), sfreq=SFREQ, cap_s=CAP_S, dropout=DROPOUT,
        trained_on='Sleep-EDF sleep-cassette, harness train split, harness default preprocessing',
        test_bmae=st['bmae'], test_wbmae=st['wbmae']), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)

    # Load it the way the harness does, with a stand-in for the base class.
    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def run(names, Xa):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                     'sfreq': SFREQ, 'submission_dir': out,
                                     'device': DEVICE, 'n_outputs': 1})
        return m.predict(torch.tensor(Xa)).cpu().numpy()

    k = rng.choice(len(Y['test']), 256, replace=False)
    Xs = X['test'][k]
    base = run(CH, Xs)
    rev = run(CH[::-1], Xs[:, ::-1].copy())
    wide = run(list(CH) + ['horizontal', 'submental'],
               np.concatenate([Xs, rng.standard_normal((len(Xs), 2, N_TIMES)).astype('float32')], 1))
    part = run(CH[:1], Xs[:, :1])
    try:
        run(['TP9', 'AF7', 'AF8', 'TP10'], rng.standard_normal((4, 4, N_TIMES)).astype('float32'))
        muse = False
    except RuntimeError:
        muse = True

    checks = [
        ('shape and range', base.shape == (len(Xs),) and np.isfinite(base).all()
         and base.min() >= 0 and base.max() <= CAP_S, f'{base.shape}, {base.min():.0f}..{base.max():.0f} s'),
        ('same as this notebook', np.allclose(base, pt[k], atol=1e-2),
         f'max |diff| {np.abs(base - pt[k]).max():.1e} s'),
        ('order-independent', np.allclose(base, rev, atol=1e-4), 'electrodes listed in reverse'),
        ('extra electrodes ignored', np.allclose(base, wide, atol=1e-4), 'two unknown channels added'),
        ('missing electrode', np.isfinite(part).all(), f'{CH[1]} zero-filled, still runs'),
        ('no match raises', muse, 'Muse-style names: warm-up model, refuses rather than guessing'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e3:.0f} KB)')
        print('Download it from Drive and upload to Codabench (Track 3).')
    else:
        print('\nNot zipping: a check failed.')